# comparison

In [ ]:
import os, sys, importlib, time

# Drop any cached combra.metrics modules so the lazy_loader stub doesn't hand
# back a pre-edit `compare_folders`. We must clear the parent `combra.metrics`
# package too, otherwise its cached attribute survives a child reload.
for m in [k for k in list(sys.modules) if k == 'combra.metrics' or k.startswith('combra.metrics.')]:
    del sys.modules[m]

import combra.metrics.compare as _cmp
from combra.metrics.compare import compare_folders

print('cwd:', os.getcwd())
print('compare.py:', _cmp.__file__, 'mtime:', time.ctime(os.path.getmtime(_cmp.__file__)))
print('compare_folders has [WARN] patch:', '[WARN]' in __import__('inspect').getsource(compare_folders))

# Shared canonical ethalon — same parquet 4_grid_plot reads (256x256, all steps).
ethalon_path = './grid_results/o_bc_left_4x_1536_1024x1024_256x256_rgb_N360_msl5/angles_n360.parquet'
print('ethalon:', os.path.abspath(ethalon_path),
      'mtime:', time.ctime(os.path.getmtime(ethalon_path)) if os.path.exists(ethalon_path) else 'MISSING')

import pyarrow.parquet as _pq
_t = _pq.read_table(ethalon_path, columns=['meta','prep_per_step'])
_pps = _t['prep_per_step'].to_pylist()[0]
print('ethalon steps:', sorted(e['step'] for e in _pps))
print()

# Explicit parquet file paths (not folders) so compare_folders processes exactly
# one file per row — avoids duplicates when a folder contains multiple sample sizes.
GRID = './grid_results'
folder_paths = [
    # The N10000 file is what 4_grid_plot's diffit res=256 row uses; this row
    # MUST numerically match `diffit res=256 N=1000` in 4_grid_plot.
    f'{GRID}/00017-diffit-256-gpus2-batch192_N10000_msl5/angles_n1000.parquet',

    # kimg snapshots — regenerated with current combra code.
    f'{GRID}/00017-diffit-256-gpus2-batch192_kimg_004435_msl5/angles_n1000.parquet',
    f'{GRID}/00017-diffit-256-gpus2-batch192_kimg_006451_msl5/angles_n1000.parquet',
    f'{GRID}/00017-diffit-256-gpus2-batch192_kimg_008064_msl5/angles_n1000.parquet',
    f'{GRID}/00017-diffit-256-gpus2-batch192_kimg_012096_msl5/angles_n1000.parquet',
    f'{GRID}/00017-diffit-256-gpus2-batch192_kimg_016128_msl5/angles_n1000.parquet',
]
for p in folder_paths:
    print('  fake:', p, 'exists:', os.path.exists(p),
          'mtime:', time.ctime(os.path.getmtime(p)) if os.path.exists(p) else '-')
print()

# Generated parquet uses class_0/1/2; ethalon uses class_Ultra_Co*. Map by training-class index.
class_map = {
    'class_0': 'class_Ultra_Co11',
    'class_1': 'class_Ultra_Co25',
    'class_2': 'class_Ultra_Co6_2',
}

res = compare_folders(folder_paths, ethalon_path, class_map=class_map, steps=[2], coef=1000)